In [1]:
import numpy as np
import pandas as pd
import re

df = pd.read_csv('../Database SSDC 2026/Database SSDC 2026 UNZIP/tracking_student.csv')

# UNIVERSAL

## 2.1. Bentuk dan Struktur

### Jumlah baris dan kolom

In [2]:
print(f"Jumlah baris : {df.shape[0]}")
print(f"Jumlah kolom : {df.shape[1]}")

expected_columns = 11
print(f"Sesuai dokumentasi ({expected_columns})? = ", df.shape[1] == expected_columns)

Jumlah baris : 41600
Jumlah kolom : 11
Sesuai dokumentasi (11)? =  True


In [36]:
df.tail(5)

,id_tracking_student,NIM,id_tracking_company,student_name,internship_semester,company,position,jenis_penempatan,progress_student,last_update,rejection
41595,TS41596,20225636,TC11999,Dian Ningsih,3,CV Cakra Mandiri,Data Analyst,Part-time,FU 3,2025-02-07,On Progress
41596,TS41597,202022674,TC11999,Tika Suryadi,8,CV Cakra Mandiri,Data Analyst,Part-time,Finish,2025-01-30,Rejection Screening CV
41597,TS41598,20214515,TC12000,Kirana Ramadhan,7,PT Bumi Informatika,Research Assistant,Part-time,Interview User,2024-12-04,On Progress
41598,TS41599,20224480,TC12000,Irfan Fitriani,3,PT Bumi Informatika,Research Assistant,Part-time,FU 2,2024-12-31,On Progress
41599,TS41600,202224947,TC12000,Gilang Mahendra,3,PT Bumi Informatika,Research Assistant,Part-time,Rejected,2025-02-09,Rejection Final Interview


### Kolom tak terduga dan hilang

In [5]:
kolom_aktual = set(df.columns)
kolom_dokumentasi = {'id_tracking_student', 'NIM', 'id_tracking_company', 'student_name', 'internship_semester', 'company', 'position', 'jenis_penempatan', 'progress_student', 'last_update', 'rejection'}

tidak_terduga = kolom_aktual - kolom_dokumentasi
hilang = kolom_dokumentasi - kolom_aktual

print("Kolom tak terduga (ada di data, tidak ada di dokumentasi):", tidak_terduga)
print("Kolom hilang (ada di dokumentasi, tidak ada di data)     :", hilang)

Kolom tak terduga (ada di data, tidak ada di dokumentasi): set()
Kolom hilang (ada di dokumentasi, tidak ada di data)     : set()


### Header

In [6]:
preview = pd.read_csv("../Database SSDC 2026/Database SSDC 2026 UNZIP/tracking_student.csv", header=None, nrows=5)
print(preview)

# kalau ternyata ada baris judul/kosong di atas header asli, load ulang dengan skiprows:
# df = pd.read_csv("nama_file.csv", skiprows=1)

                    0          1                    2               3   \
0  id_tracking_student        NIM  id_tracking_company    student_name   
1               TS0001  202018732                TC001   Teguh Maulana   
2               TS0002  202312115                TC001    Dwi Pangestu   
3               TS0003   20231598                TC001  Vina Ramadhani   
4               TS0004  202117594                TC001   Nadia Hartono   

                    4              5                           6   \
0  internship_semester        company                    position   
1                    7  PT Prima Data  Project Coordinator Intern   
2                    2  PT Prima Data  Project Coordinator Intern   
3                    2  PT Prima Data  Project Coordinator Intern   
4                    5  PT Prima Data  Project Coordinator Intern   

                 7                 8            9                          10  
0  jenis_penempatan  progress_student  last_update          

## 2.2. Tipe data

### Tipe data pandas

-->> nim nya int, harusnya objek

In [7]:
df.dtypes

id_tracking_student      str
NIM                    int64
id_tracking_company      str
student_name             str
internship_semester    int64
company                  str
position                 str
jenis_penempatan         str
progress_student         str
last_update              str
rejection                str
dtype: object

### Parse date

2023-2025 --> AMAN

In [8]:
kolom_tanggal = ["last_update"] 

for kolom in kolom_tanggal:
    parsed = pd.to_datetime(df[kolom], errors="coerce")
    gagal_parse = parsed.isna().sum() - df[kolom].isna().sum()
    print(f"Kolom '{kolom}':")
    print(f"  Gagal di-parse jadi tanggal : {gagal_parse}")
    if parsed.notna().any():
        print(f"  Range tanggal               : {parsed.min()}  s/d  {parsed.max()}")

Kolom 'last_update':
  Gagal di-parse jadi tanggal : 0
  Range tanggal               : 2023-02-08 00:00:00  s/d  2025-05-17 00:00:00


## 2.3. Missing Values

### Persentase isnull

Aman karena draft berarti belum dikirimkan
berarti send_date kosong -->> belum ada pengiriman (karena masih on proses)
berarti list_nim kosong -->> belum ada mahasiswa yang mengirimkan (karena on proses)

In [9]:
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(null_pct[null_pct > 0].sort_values(ascending=False))

Series([], dtype: float64)


### Null tersamar

-->> tidak ada null tersamar dan tidak ada double information (penulisan) value

In [11]:
print(df.nunique(dropna=False))

id_tracking_student    41600
NIM                    10174
id_tracking_company    11402
student_name            4784
internship_semester       10
company                 1488
position                  74
jenis_penempatan           3
progress_student          12
last_update              821
rejection                  7
dtype: int64


In [12]:
print(df.nunique(dropna=True))

id_tracking_student    41600
NIM                    10174
id_tracking_company    11402
student_name            4784
internship_semester       10
company                 1488
position                  74
jenis_penempatan           3
progress_student          12
last_update              821
rejection                  7
dtype: int64


In [14]:
token_null_tersamar = ["", " ", "-", "N/A", "NA", "na", "null", "None", "TBD",
                        "Belum ada", "belum ada", "?", "--"]

kolom_high_cardinality = ["id_tracking_student", "NIM", "id_tracking_company",
                           "student_name", "company"]

for kolom in kolom_high_cardinality:
    ditemukan = df[kolom].astype(str).str.strip().isin(token_null_tersamar)
    if ditemukan.sum() > 0:
        print(f"Kolom '{kolom}': {ditemukan.sum()} baris null tersamar")
        print(df.loc[ditemukan, kolom].value_counts())
    else:
        print(f"Kolom '{kolom}': aman, tidak ditemukan null tersamar")

Kolom 'id_tracking_student': aman, tidak ditemukan null tersamar
Kolom 'NIM': aman, tidak ditemukan null tersamar
Kolom 'id_tracking_company': aman, tidak ditemukan null tersamar
Kolom 'student_name': aman, tidak ditemukan null tersamar
Kolom 'company': aman, tidak ditemukan null tersamar


In [15]:
for kolom in df.columns:
    n_unique = df[kolom].nunique(dropna=False)
    if n_unique <= 150:
        print(f"\n{'='*50}")
        print(f"Kolom: '{kolom}'  ({n_unique} distinct values)")
        print(df[kolom].value_counts(dropna=False))
    else:
        print(f"\nKolom '{kolom}' dilewati ({n_unique} distinct values, terlalu banyak untuk dilihat manual)")


Kolom 'id_tracking_student' dilewati (41600 distinct values, terlalu banyak untuk dilihat manual)

Kolom 'NIM' dilewati (10174 distinct values, terlalu banyak untuk dilihat manual)

Kolom 'id_tracking_company' dilewati (11402 distinct values, terlalu banyak untuk dilihat manual)

Kolom 'student_name' dilewati (4784 distinct values, terlalu banyak untuk dilihat manual)

Kolom: 'internship_semester'  (10 distinct values)
internship_semester
5     8185
3     7017
7     6158
2     5739
4     4055
6     4054
9     2956
8     2093
11     688
10     655
Name: count, dtype: int64

Kolom 'company' dilewati (1488 distinct values, terlalu banyak untuk dilihat manual)

Kolom: 'position'  (74 distinct values)
position
Data Analyst                4765
IT Support                  3308
Quality Control Staff       1387
Business Analyst            1226
Digital Marketing Intern    1209
                            ... 
Mobile Developer             210
DevOps Engineer              191
Data Science Intern 

## 2.4. Duplikasi

### Fully duplicated rows

-->> aman

In [18]:
jumlah_duplikat = df.duplicated().sum()
print(f"Duplikat baris penuh: {jumlah_duplikat}")

if jumlah_duplikat > 0:
    print(df[df.duplicated(keep=False)].sort_values(by=df.columns[0]))

Duplikat baris penuh: 0


### Primary key duplicated

-->> aman

In [20]:
pk = "id_tracking_student"  # ganti sesuai PK tabel ini

jumlah_dup_pk = df[pk].duplicated().sum()
print(f"Duplikat pada PK ('{pk}'): {jumlah_dup_pk}")

if jumlah_dup_pk > 0:
    print(df[df[pk].duplicated(keep=False)].sort_values(pk))

Duplikat pada PK ('id_tracking_student'): 0


### semanthic duplicate

-->> 1 nama untuk beberapa NIM 

In [22]:
# Cek konsistensi NIM <-> student_name di tracking_student

print(df.columns.tolist())  # pastikan nama kolom persis: NIM/nim, student_name

# 1. Nama sama, tapi NIM berbeda
cek_nama = df.groupby("student_name")["NIM"].nunique()
kandidat_1 = cek_nama[cek_nama > 1]
print("student_name sama, tapi NIM berbeda:")
print(kandidat_1)

# 2. NIM sama, tapi nama berbeda (indikasi typo penulisan nama)
cek_nim = df.groupby("NIM")["student_name"].nunique()
kandidat_2 = cek_nim[cek_nim > 1]
print("\nNIM sama, tapi student_name berbeda:")
print(kandidat_2)

# 3. Kalau ada temuan, tampilkan baris asli untuk investigasi manual
if len(kandidat_1) > 0:
    print("\n--- Detail kasus: nama sama, NIM beda ---")
    for nama in kandidat_1.index:
        print(df[df["student_name"] == nama][["NIM", "student_name"]])

if len(kandidat_2) > 0:
    print("\n--- Detail kasus: NIM sama, nama beda ---")
    for nim in kandidat_2.index:
        print(df[df["NIM"] == nim][["NIM", "student_name"]])

if len(kandidat_1) == 0 and len(kandidat_2) == 0:
    print("\nAman — tidak ditemukan inkonsistensi antara NIM dan student_name.")

['id_tracking_student', 'NIM', 'id_tracking_company', 'student_name', 'internship_semester', 'company', 'position', 'jenis_penempatan', 'progress_student', 'last_update', 'rejection']
student_name sama, tapi NIM berbeda:
student_name
Aditya Anggraeni        2
Aditya Budiman          4
Aditya Firmansyah       2
Aditya Fitriani         3
Aditya Gunawan          2
Aditya Handoko          2
Aditya Mahendra         4
Aditya Maulana          2
Aditya Nugraha          4
Aditya Nugroho          3
Aditya Pangestu         3
Aditya Permana          2
Aditya Prabowo          2
Aditya Putra            3
Aditya Ramadhan         3
Aditya Ramadhani        2
Aditya Safitri          6
Aditya Santoso          2
Aditya Setiawan         3
Aditya Sulistyo         3
Aditya Suryadi          2
Aditya Syahputra        3
Aditya Utami            3
Aditya Wibowo           2
Aditya Wulandari        2
Ahmad Adriansyah        2
Ahmad Anggara           2
Ahmad Cahyani           2
Ahmad Cahyono           2
Ahmad Dewi  

In [26]:
print("Jumlah distinct NIM:", df["NIM"].nunique())
print("Jumlah distinct student_name:", df["student_name"].nunique())
print("Total baris:", len(df))

Jumlah distinct NIM: 10174
Jumlah distinct student_name: 4784
Total baris: 41600


In [27]:
print("\nDistribusi jumlah NIM unik per nama (top 10 nama paling sering dipakai bareng NIM beda):")
print(df.groupby("student_name")["NIM"].nunique().sort_values(ascending=False).head(10))


Distribusi jumlah NIM unik per nama (top 10 nama paling sering dipakai bareng NIM beda):
student_name
Eka Adriansyah       9
Hesti Cahyani        8
Eka Sulistyo         8
Gilang Adriansyah    8
Nadya Rahmawati      7
Nadya Saputra        7
Ilham Puspitasari    7
Ratna Prabowo        7
Eka Hartono          7
Dimas Anggraeni      7
Name: NIM, dtype: int64


In [28]:
distribusi_panjang_nim = df["NIM"].astype(str).str.len().value_counts()
print(distribusi_panjang_nim)

NIM
9    25081
8    16519
Name: count, dtype: int64


In [ ]:
df_tc = pd.read_csv('../Database SSDC 2026/Database SSDC 2026 UNZIP/tracking_company.csv')
df_company    = pd.read_csv("../Database SSDC 2026/Database SSDC 2026 UNZIP/company.csv")

# langkah 1: gabungkan tracking_student -> tracking_company untuk dapat id_company
merged = df.merge(df_tc[["id_tracking_company", "id_company"]], on="id_tracking_company", how="left")

# langkah 2: gabungkan lagi ke tabel company untuk dapat company_name resmi
merged = merged.merge(df_company[["id_company", "company_name"]], on="id_company", how="left")

# langkah 3: bandingkan company (di tracking_student) vs company_name (resmi)
tidak_cocok = merged[merged["company"].str.strip().str.lower() != merged["company_name"].str.strip().str.lower()]
print(f"Baris dengan nama company tidak konsisten: {len(tidak_cocok)}")
print(tidak_cocok[["company", "company_name"]].drop_duplicates())

Baris dengan nama company tidak konsisten: 0
Empty DataFrame
Columns: [company, company_name]
Index: []


## Kategorikal

In [29]:
pd.set_option("display.max_rows", None)

### Value counts
-->> udah di cek di sebelumnya juga kok aman

### Variasi penulisan

In [31]:
kolom_kategorikal = {'id_tracking_student', 'NIM', 'id_tracking_company', 'student_name', 'company', 'position', 'jenis_penempatan', 'progress_student', 'last_update', 'rejection'}
def cek_variasi_penulisan(kolom):
    nilai_unik = df[kolom].dropna().unique()
    versi_bersih = {}
    for v in nilai_unik:
        key = str(v).strip().lower()
        versi_bersih.setdefault(key, []).append(v)

    hasil = {k: v for k, v in versi_bersih.items() if len(v) > 1}
    print(f"\nKolom '{kolom}' — kandidat variasi penulisan sama makna:")
    if hasil:
        for k, v in hasil.items():
            print(f"  {v}")
    else:
        print("  Tidak ditemukan.")

for kolom in kolom_kategorikal:
    cek_variasi_penulisan(kolom)


Kolom 'position' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.

Kolom 'NIM' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.

Kolom 'progress_student' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.

Kolom 'rejection' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.

Kolom 'id_tracking_student' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.

Kolom 'id_tracking_company' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.

Kolom 'student_name' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.

Kolom 'company' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.

Kolom 'jenis_penempatan' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.

Kolom 'last_update' — kandidat variasi penulisan sama makna:
  Tidak ditemukan.


### Kemiripan

In [32]:
from rapidfuzz import fuzz

def cek_typo_mirip(kolom, threshold=85, batas_unik=150):
    nilai_unik = df[kolom].dropna().unique().tolist()
    if len(nilai_unik) > batas_unik:
        print(f"\nKolom '{kolom}' dilewati untuk fuzzy check ({len(nilai_unik)} distinct, terlalu banyak)")
        return
    print(f"\nKolom '{kolom}' — pasangan mirip (similarity >= {threshold}):")
    ditemukan = False
    for i in range(len(nilai_unik)):
        for j in range(i+1, len(nilai_unik)):
            skor = fuzz.ratio(str(nilai_unik[i]).lower(), str(nilai_unik[j]).lower())
            if skor >= threshold:
                print(f"  '{nilai_unik[i]}'  <->  '{nilai_unik[j]}'   (skor: {skor})")
                ditemukan = True
    if not ditemukan:
        print("  Tidak ditemukan.")

for kolom in kolom_kategorikal:
    cek_typo_mirip(kolom)


Kolom 'position' — pasangan mirip (similarity >= 85):
  'Consultant Intern'  <->  'IT Consultant Intern'   (skor: 91.89189189189189)

Kolom 'NIM' dilewati untuk fuzzy check (10174 distinct, terlalu banyak)

Kolom 'progress_student' — pasangan mirip (similarity >= 85):
  Tidak ditemukan.

Kolom 'rejection' — pasangan mirip (similarity >= 85):
  Tidak ditemukan.

Kolom 'id_tracking_student' dilewati untuk fuzzy check (41600 distinct, terlalu banyak)

Kolom 'id_tracking_company' dilewati untuk fuzzy check (11402 distinct, terlalu banyak)

Kolom 'student_name' dilewati untuk fuzzy check (4784 distinct, terlalu banyak)

Kolom 'company' dilewati untuk fuzzy check (1488 distinct, terlalu banyak)

Kolom 'jenis_penempatan' — pasangan mirip (similarity >= 85):
  Tidak ditemukan.

Kolom 'last_update' dilewati untuk fuzzy check (821 distinct, terlalu banyak)


## 2.6. ID Format

### Penulisan ID

banyak banget ya -->> GA AMAN

In [41]:
kolom_id_pattern = {
    "id_tracking_student": r"^TS\d{3}$",
    "id_tracking_company": r"^TC\d{3}$"
    # sesuaikan pattern lain sesuai dokumentasi (SS, TS, dst)
}

for kolom, pattern in kolom_id_pattern.items():
    if kolom in df.columns:
        tidak_sesuai = df[~df[kolom].astype(str).str.match(pattern)]
        print(f"Kolom '{kolom}': {len(tidak_sesuai)} baris tidak sesuai pattern '{pattern}'")
        if len(tidak_sesuai) > 0:
            print(tidak_sesuai[kolom].unique()[:10])

Kolom 'id_tracking_student': 41600 baris tidak sesuai pattern '^TS\d{3}$'
<StringArray>
['TS0001', 'TS0002', 'TS0003', 'TS0004', 'TS0005', 'TS0006', 'TS0007',
 'TS0008', 'TS0009', 'TS0010']
Length: 10, dtype: str
Kolom 'id_tracking_company': 38220 baris tidak sesuai pattern '^TC\d{3}$'
<StringArray>
['TC1000', 'TC1001', 'TC1002', 'TC1003', 'TC1004', 'TC1005', 'TC1006',
 'TC1007', 'TC1008', 'TC1009']
Length: 10, dtype: str


### Leading Zero

-->> prefiks aman

In [42]:
for kolom in kolom_id_pattern.keys():
    if kolom in df.columns:
        contoh = df[kolom].astype(str).unique()[:5]
        print(f"\nKolom '{kolom}' — contoh nilai: {contoh}")
        # cek apakah ada nilai yang cuma angka tanpa prefix huruf & tanpa leading zero
        murni_angka = df[kolom].astype(str).str.match(r"^\d+$")
        if murni_angka.sum() > 0:
            print(f"  [WARNING] {murni_angka.sum()} baris berupa angka murni (kemungkinan leading zero/prefix hilang)")


Kolom 'id_tracking_student' — contoh nilai: <StringArray>
['TS0001', 'TS0002', 'TS0003', 'TS0004', 'TS0005']
Length: 5, dtype: str

Kolom 'id_tracking_company' — contoh nilai: <StringArray>
['TC001', 'TC002', 'TC003', 'TC004', 'TC005']
Length: 5, dtype: str


### inkonsistensi panjang id

expected:
string id_company = 4
string id_talent_req = 5
string id_tracking_company = 5

-->> GA AMAN

In [43]:
for kolom in kolom_id_pattern.keys():
    if kolom in df.columns:
        panjang = df[kolom].astype(str).str.len().value_counts()
        print(f"\nKolom '{kolom}' — distribusi panjang string:")
        print(panjang)


Kolom 'id_tracking_student' — distribusi panjang string:
id_tracking_student
7    31601
6     9999
Name: count, dtype: int64

Kolom 'id_tracking_company' — distribusi panjang string:
id_tracking_company
6    31329
7     6891
5     3380
Name: count, dtype: int64


### Whitespace

-->> AMAN

In [46]:
for kolom in ["id_tracking_student", "id_tracking_company", "NIM"]:
    asli = df[kolom].astype(str)
    ws = asli[asli != asli.str.strip()]
    print(f"Kolom '{kolom}': {len(ws)} baris dengan whitespace")

Kolom 'id_tracking_student': 0 baris dengan whitespace
Kolom 'id_tracking_company': 0 baris dengan whitespace
Kolom 'NIM': 0 baris dengan whitespace
